# CoachBot AI exploration companion
Engineering demonstrations only. Your observations and conclusions: [MY INPUT NEEDED].

Use a Python environment with requirements.txt installed, in a notebook editor such as VS Code. No notebook server is required by the app. Live execution is disabled by default. Select your project environment as the kernel. Do not save API keys or private model outputs into the notebook.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd()
if not (root / "core").is_dir():
    root = root.parent
assert (root / "core").is_dir(), "Open this notebook from the project or notebooks folder"
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from core.groq_client import GroqClient, CallBudget, SetupError
from core.schemas import CoachingResponse
from eval.eval_runner import load_cases, build_jobs, run_jobs
cases = load_cases(root / "eval/cases.synthetic.json")
ALLOW_LIVE_CALLS = False

## API connectivity test
After configuring st.secrets and config.MODEL_NAME, deliberately enable ALLOW_LIVE_CALLS to send the synthetic baseline through the Groq core client. Hard ceiling: one generation attempt; repairs/retries cannot exceed it. The cached request costs zero calls on repeat. No real athlete information is used.

In [ ]:
job = build_jobs(cases, subset=1)[0]
if ALLOW_LIVE_CALLS:
    try:
        with GroqClient.from_streamlit(cache_namespace="notebook-connectivity", sleep_seconds=2) as client:
            result = client.generate(job.preflight.request, budget=CallBudget(1))
            print({"status": result.status, "calls": result.calls_made, "cache_hit": result.cache_hit})
    except SetupError as exc:
        print(str(exc))
else:
    print("Connectivity request skipped; zero API calls.")

## Parameter-sweep demonstration
Three output-token limits with one temperature/top_p pair: three jobs, typically three uncached calls, at most eighteen including retries/repairs without a cap. This demo uses a shared three-call ceiling. Examine summary.csv and tuning.csv after a deliberate live run. Do not infer quality from output length alone.

In [ ]:
jobs = build_jobs(cases[:1], temperatures=(0.2,), top_ps=(0.8,), max_tokens=(256, 512, 1024))
print([j.parameters.model_dump() for j in jobs])
if ALLOW_LIVE_CALLS:
    try:
        print(run_jobs(jobs, root / "eval/results/notebook-demo", max_calls=3, sleep_seconds=2))
    except SetupError as exc:
        print(str(exc))
else:
    print("Sweep planned only; zero API calls.")

## Schema-validation demonstration
Reuse the core-generated local safety notice; remove its rationale to demonstrate a rejected schema. This is a structural check, not a quality judgment.

In [ ]:
from pydantic import ValidationError
local = build_jobs([cases[-1]], subset=1)[0].preflight.local_response
valid = CoachingResponse.model_validate_json(local.model_dump_json())
print("Complete local notice validates:", bool(valid.recommendations))
broken = valid.model_dump()
del broken["recommendations"][0]["rationale"]
try:
    CoachingResponse.model_validate(broken)
except ValidationError:
    print("Missing rationale correctly rejected.")

## My notes
- Connectivity evidence: [MY INPUT NEEDED]
- Parameter observations, including length and completeness: [MY INPUT NEEDED]
- Quality-checklist scores and reasons: [MY INPUT NEEDED]

Clear cell outputs before committing if you enable live calls. Do not publish eval/results or response caches without manual review. English is the app default; the separate Tamil synthetic case tests the explicit language-selection path.